# lcd-ru — Colab worker

Run this notebook and leave the tab open; jobs are queued from the local machine (`python colab/jobs.py add ...`).

1. Runtime → Change runtime type → **L4** or **A100** GPU (bf16 needed; T4 is refused by the setup).
2. 🔑 Secrets (Notebook access on): `KAGGLE_API_TOKEN` = contents of `~/.kaggle/access_token`; `HF_TOKEN` = a Hugging Face read token (optional for Qwen; required for Llama-3.2-3B; avoids HF rate limits).
3. Runtime → Run all. No Google Drive: everything persistent is a private Kaggle dataset (`lcd-ru-*`).

The worker restores its history from `kirillkholkin/lcd-ru-colab-status-a`, sets the session up (pip, `data` artifact), polls `kirillkholkin/lcd-ru-jobs` every 2 min, runs jobs and publishes status, log tails and small results. Training saves checkpoints to `lcd-ru-runs-<name>` every N minutes, so after a disconnect just Run all again. After 45 min without jobs the runtime is released.

In [ ]:
from google.colab import userdata
import os, subprocess
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN').strip()
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()
except Exception:
    print('no HF_TOKEN secret: fine for Qwen, required for meta-llama/Llama-3.2-3B')
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
tok = os.path.expanduser('~/.kaggle/access_token')
open(tok, 'w').write(os.environ['KAGGLE_API_TOKEN']); os.chmod(tok, 0o600)
subprocess.run('pip install -q -U "kaggle>=2.0"', shell=True, check=True)
subprocess.run('rm -rf /content/code && kaggle datasets download kirillkholkin/lcd-ru-code -p /content/code --unzip -q',
               shell=True, check=True)
print('code version', open('/content/code/CODE_VERSION').read().strip(), 'git', open('/content/code/GIT_HASH').read().strip())
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; nproc; free -g | head -2

In [ ]:
%run /content/code/colab/worker.py --worker A